# Clase 9 — RAG Lab en vivo

Hoy vamos a construir, romper y mejorar un RAG usando PDFs.

**Flujo:** PDF → extracción → metadata → chunking → embeddings → Chroma → retrieval → prompt → LLM → evaluación

> Regla del laboratorio: cambiar una variable por vez, repetir las mismas pruebas y comparar.

## 0. Preparación

La notebook espera:

```text
RAG-vacaciones/
├── Clase_9_RAG_Lab_Alumno_PDFs.ipynb
├── .env
└── rag_lab_pdfs/
    ├── politica_vacaciones_2024.pdf
    ├── politica_vacaciones_2026.pdf
    └── politica_home_office_2026.pdf
```

En `.env`:

```env
COHERE_API_KEY=TU_API_KEY
```

Instalación recomendada:

```bash
python -m pip install pandas chromadb cohere pypdf python-dotenv ipykernel
```

In [ ]:
import os
import re
import sys
from pathlib import Path

import pandas as pd
import chromadb
import cohere

from dotenv import load_dotenv
from pypdf import PdfReader

print("Python:", sys.executable)

In [ ]:
load_dotenv()

COHERE_API_KEY = os.getenv("COHERE_API_KEY")

if not COHERE_API_KEY:
    print("⚠️ Falta COHERE_API_KEY")
    co = None
else:
    print("✅ API Key detectada")
    co = cohere.ClientV2(COHERE_API_KEY)

# 1. Dataset de laboratorio

Vamos a usar PDFs reales de laboratorio en lugar de textos pegados en Python.

### Challenge

Cargar todos los PDFs de la carpeta `rag_lab_pdfs` y mostrar cuántos encontró.

In [ ]:
PDF_FOLDER = Path("rag_lab_pdfs")

# TODO:
# 1. verificar que la carpeta exista
# 2. buscar todos los .pdf
# 3. ordenarlos
# 4. imprimir sus nombres

pdf_files = []

print("📄 PDFs encontrados:", len(pdf_files))

## 1.1 Extraer texto por página

### Challenge

Leer cada PDF con `PdfReader` y construir una estructura así:

```python
{
    "source": "archivo.pdf",
    "pages": [
        {"page": 1, "text": "..."},
        {"page": 2, "text": "..."}
    ]
}
```

In [ ]:
documents = []

# TODO:
# recorrer pdf_files
# abrir cada PDF
# recorrer sus páginas
# extraer texto
# guardar source + page + text

print("✅ Documentos cargados:", len(documents))

In [ ]:
# Inspección
# TODO: mostrar el nombre del primer documento
# y los primeros 500 caracteres de su primera página

### Pregunta

¿Por qué conviene inspeccionar el texto extraído antes de generar embeddings?

# 2. Metadata

Vamos a inferir metadata básica desde el nombre del archivo.

### Challenge

Crear una función `infer_metadata(filename)` que devuelva:

```python
{
    "year": 2026,
    "department": "RRHH",
    "country": "AR"
}
```

Sugerencias:
- si contiene `vacaciones` o `licencias` → RRHH
- si contiene `reintegro` → Finanzas
- si contiene `home_office`, `beneficios` u `onboarding` → People

In [ ]:
def infer_metadata(filename: str) -> dict:
    # TODO:
    # 1. detectar año con regex
    # 2. inferir departamento
    # 3. devolver country="AR"
    pass


# TODO: agregar metadata a cada documento

# 3. Challenge 1 — Chunking

Queremos preservar:

- documento
- página
- chunk_index
- año
- área
- país

También vamos a separar:

- `text`
- `embedding_text`

In [ ]:
def clean_text(text: str) -> str:
    # TODO: limpiar espacios repetidos
    pass


def simple_chunk(text: str, chunk_size=600, overlap=100):
    # TODO:
    # 1. limpiar texto
    # 2. cortar en fragmentos de chunk_size
    # 3. aplicar overlap
    # 4. devolver lista de strings
    pass

In [ ]:
def build_chunks(documents, chunk_size=600, overlap=100):
    chunks = []

    # TODO:
    # recorrer documentos
    # recorrer páginas
    # crear chunks
    # conservar metadata
    # crear embedding_text enriquecido

    return chunks


chunks = build_chunks(documents)

print("Chunks:", len(chunks))

### Pregunta

¿Por qué puede ayudar agregar el nombre del documento o el año al texto que se embebe?

# 4. Challenge 2 — Embeddings

Usaremos:

- `search_document` para chunks
- `search_query` para consultas

In [ ]:
EMBED_MODEL = "embed-multilingual-v3.0"


def embed_documents(texts):
    # TODO: usar co.embed con input_type="search_document"
    pass


def embed_query(query):
    # TODO: usar co.embed con input_type="search_query"
    pass

In [ ]:
# TODO:
# generar embeddings para todos los embedding_text
# imprimir cantidad y dimensión

embeddings = []

# 5. Challenge 3 — Chroma

### Challenge

Crear una colección con distancia coseno e indexar:

- ids
- embeddings
- documents
- metadatas

In [ ]:
client = chromadb.Client()

COLLECTION_NAME = "pia_rag_lab"

# TODO:
# borrar colección si ya existe
# crear colección con cosine
# agregar chunks

collection = None

# 6. Challenge 4 — Retrieval

Antes de usar el LLM, tenemos que mirar qué recupera el buscador.

### Challenge

Crear `retrieve(query, k=3, where=None)` y devolver un DataFrame con:

- rank
- source
- page
- year
- department
- distance
- text

In [ ]:
def retrieve(query, k=3, where=None):
    # TODO:
    # 1. generar embedding de la query
    # 2. consultar Chroma
    # 3. transformar resultados a DataFrame
    pass

In [ ]:
query = "¿Cuántos días de vacaciones tiene una persona con 7 años de antigüedad?"

# TODO:
# probar retrieve con k=5

### Preguntas

- ¿Aparece la versión correcta?
- ¿En qué posición?
- ¿Qué documentos compiten con ella?
- ¿Qué tan cerca están las distancias?

# 7. Challenge 5 — Top-K

Probar exactamente la misma consulta con:

- K = 1
- K = 3
- K = 5

### Pregunta

¿Qué cambia entre recall y ruido?

In [ ]:
test_query = "¿Cuántos días de vacaciones tiene alguien con 7 años de antigüedad?"

# TODO:
# ejecutar retrieve para K=1, 3 y 5

# 8. Challenge 6 — Prompting + RAG

Vamos a comparar:

1. Prompt mínimo
2. Prompt grounded

In [ ]:
GEN_MODEL = "command-a-03-2025"


def build_context(results_df):
    # TODO:
    # combinar texto + fuente + página
    pass

In [ ]:
def generate_baseline(query, k=3):
    # TODO:
    # retrieve
    # build_context
    # co.chat con prompt mínimo
    pass

In [ ]:
def generate_grounded(query, k=3, where=None):
    # TODO:
    # retrieve
    # build_context
    # system prompt con reglas:
    # - usar sólo documentación recuperada
    # - no completar con conocimiento externo
    # - priorizar versión más reciente
    # - fallback si no alcanza evidencia
    # - citar fuente y página
    pass

In [ ]:
query = "¿Cuántos días de vacaciones tiene una persona con 7 años de antigüedad?"

# TODO:
# comparar baseline vs grounded

# 9. Challenge 7 — Fuera de contexto

Probar:

- ¿Cuál es la capital de Francia?
- ¿Quién es Harry Potter?
- ¿Cuál es el sueldo de Ana Pérez?

### Pregunta

¿Por qué el vector store devuelve resultados aunque la pregunta no pertenezca al corpus?

In [ ]:
out_of_context_queries = [
    "¿Cuál es la capital de Francia?",
    "¿Quién es Harry Potter?",
    "¿Cuál es el sueldo de Ana Pérez?"
]

# TODO:
# ejecutar generate_grounded para cada consulta
# inspeccionar también el retrieval

# 10. Challenge 8 — Metadata Filtering

Comparar la misma consulta:

1. sin filtro
2. con `where={"year": 2026}`

### Pregunta

¿Qué problema resuelve metadata filtering que la similitud vectorial no resuelve?

In [ ]:
vacation_query = "¿Cuántos días de vacaciones corresponden entre 5 y 10 años?"

# TODO:
# comparar retrieval sin filtro vs sólo year=2026

# 11. Challenge 9 — Evaluación de retrieval

Vamos a crear un pequeño conjunto de evaluación.

El objetivo es comprobar si la **fuente esperada aparece dentro de Top-K**.

In [ ]:
evaluation_set = [
    {
        "query": "¿Cuántos días de vacaciones tiene alguien con 7 años de antigüedad?",
        "expected_source": "politica_vacaciones_2026.pdf"
    },
    {
        "query": "¿Con cuánta anticipación debo pedir vacaciones?",
        "expected_source": "politica_vacaciones_2026.pdf"
    },
    {
        "query": "¿Cuántos días por semana puedo trabajar remoto?",
        "expected_source": "politica_home_office_2026.pdf"
    }
]

In [ ]:
def evaluate_retrieval(eval_set, k=3):
    # TODO:
    # ejecutar retrieve
    # verificar si expected_source aparece en top-k
    # guardar hit y rank
    # devolver DataFrame
    pass

In [ ]:
# TODO:
# calcular resultado para K=1, K=3 y K=5
# comparar hit rate

# 12. Bonus — Chunk size

Comparar:

- 300 / overlap 50
- 600 / overlap 100
- 1000 / overlap 150

### Pregunta

¿Por qué no podemos decidir el mejor `chunk_size` sólo mirando cuántos chunks genera?

In [ ]:
# TODO:
# construir chunks con distintas configuraciones
# comparar cantidad y longitud promedio

# 13. Cierre

Cuando una respuesta RAG falla:

1. documento
2. extracción
3. chunk
4. embedding
5. retrieval
6. metadata
7. prompt
8. generación

> La pregunta útil es: **¿qué evidencia demuestra en qué etapa falló?**